# Part 1 — Zero-Shot Mutation Scoring with ESM-2

In this notebook, we use ESM-2 **without any training** to predict the effects of mutations in TEM-1 β-lactamase.

**You will:**
1. Load ESM-2 and understand its outputs
2. Score specific TEM-1 mutations via masked marginal likelihood
3. Compute a per-position constraint profile and check whether it recovers the active site
4. Compare ESM-2 predictions to experimental fitness data from Firnberg et al. (2014)

**Runtime:** ~10 minutes on any laptop. No GPU required.

> Background and context are in the [README](./README.md). Start there if you haven't already.

## 1. Setup

Install dependencies (Colab) or skip if running locally with the env already set up.

In [ ]:
# Standard library
import re

# Third-party libraries
# Note: On Colab, uncomment the next line:
# !pip install matplotlib numpy pandas requests scipy torch transformers -q
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import torch
from scipy.stats import spearmanr
from transformers import AutoModelForMaskedLM, AutoTokenizer

In [ ]:
# Device detection (works on CPU, CUDA, or Apple Silicon MPS)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

## 2. The TEM-1 sequence

We use the mature TEM-1 sequence (without the signal peptide), 263 residues. We'll use Ambler numbering (the convention in β-lactamase literature) — see helpers below.

In [ ]:
def fetch_tem1_sequence():
    """Fetch the canonical TEM-1 mature sequence from UniProt (P62593, residues 24-286)."""
    url = "https://rest.uniprot.org/uniprotkb/P62593.fasta"
    fasta = requests.get(url).text
    return "".join(fasta.split("\n")[1:])

In [ ]:
TEM1_FULL = fetch_tem1_sequence()
assert len(TEM1_FULL) == 286, "Full sequence length is not 286"
print(f"Unprocessed sequence: {len(TEM1_FULL)} residues, starts with {TEM1_FULL[:30]}…")

In [ ]:
# Mature TEM-1 (residues 24-286 of the precursor) - the biologically functional protein

# Numbering: the literature and ProteinGym both use UniProt numbering on the
# precursor (1-indexed, position 24 = first residue of mature protein = signal peptide).
SIGNAL_PEPTIDE_LENGTH = 23


def uniprot_to_index(pos):
    """UniProt position (1-indexed on full precursor) -> 0-indexed on mature sequence."""
    return pos - SIGNAL_PEPTIDE_LENGTH - 1


TEM1_MATURE = TEM1_FULL[SIGNAL_PEPTIDE_LENGTH:]
assert len(TEM1_MATURE) == 263, "Mature sequence length is not 263"

print(f"Mature sequence: {len(TEM1_MATURE)} residues, starts with {TEM1_MATURE[:30]}…")

In [ ]:
# TEM-1 active site (catalytic residues + binding sites) (Ambler numbering)
ACTIVE_SITE = {
    "S70": 70,  # catalytic nucleophile serine
    "K73": 73,  # general base
    "S130": 130,  # proton donor
    "N132": 132,  # oxyanion hole
    "E166": 166,  # general base for deacylation
    "K234": 234,  # binding site (binds substrate carboxylate)
    "S235": 235,  # binding site
    "G236": 236,  # binding site
}

We use the mature TEM-1 sequence (263 residues, signal peptide removed). 
We use **Ambler numbering** throughout — the convention used in nearly all 
β-lactamase literature — where the catalytic serine is S70, the famous 
global suppressor is M182T, etc. Note: Ambler is offset by +2 from UniProt numbering, so the same residues are S68 and M180 in UniProt. ProteinGym uses UniProt numbering — its M180T row is the literature's M182T

In [ ]:
# Biology:
#   Same residue, two different position labels:
#   UniProt position = Ambler position - 2
# Examples: S70 (Ambler) = S68 (UniProt), M182 (Ambler) = M180 (UniProt)
AMBLER_TO_UNIPROT_OFFSET = 2


def ambler_to_uniprot(pos):
    """Convert Ambler position to the equivalent UniProt position (both 1-indexed)."""
    return pos - AMBLER_TO_UNIPROT_OFFSET


def ambler_to_index(pos):
    """Convert Ambler position to 0-indexed position in the mature sequence."""
    return uniprot_to_index(ambler_to_uniprot(pos))

In [ ]:
# Verify:
for name, pos in ACTIVE_SITE.items():
    actual = TEM1_MATURE[ambler_to_index(pos)]
    expected = name[0]
    print(f"{name} = {actual} (expected {expected}) {'✓' if actual == expected else '✗'}")
print("All catalytic residue checks passed ✓")

## 3. Load ESM-2

Now we load ESM-2, the protein language model we will use to score mutations on the TEM-1 sequence above. 
We use the small 35M-parameter variant — fast, laptop-friendly, and surprisingly capable for mutation scoring.
The model files (~140MB) download once to `~/.cache/huggingface/` and are reused on subsequent runs.

Conventions of the name "esm2_t12_35M_UR50D":
- esm2 = Evolutionary Scale Modeling.  
- t12 = 12 transformer layers (similar to BERT).  
- 35M = 35 million parameters.
- UR50 = UniRef50, a clustered version of Uniprot at 50% (D = filtered for Diversity).

The next lines:
1) load the tokenizer (which converts amino acid strings to integer IDs the model understands) 
2) load the model with a Masked Language Model (MLM) head — used for zero-shot mutation scoring:
   - .from_pretrained(MODEL_NAME) downloads and instantiates the pretrained weights.
   - .to() => moves the model to GPU/MPS for fast inference.
   - .eval() => switches off dropout — we use the model for inference, not training (instead of .train() ).

In [ ]:
MODEL_NAME = "facebook/esm2_t12_35M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {n_params / 1e6:.1f}M parameters")

## 4. Score mutations with masked marginal likelihood

The idea is simple: at the position of interest, we mask the residue and ask ESM-2 what amino acid it expects. The score for a mutation `wt → mut` is:

$$\text{score}(wt \to mut) = \log P(mut \mid \text{context}) - \log P(wt \mid \text{context})$$

- **Positive score** → the model finds the mutant *more* plausible than wild-type
- **Negative score** → the mutant is *less* plausible (likely destabilizing or function-disrupting)

This works because ESM-2's pretraining objective was exactly this: predict masked amino acids from context. We're using its learned distribution as a fitness prior.

In [ ]:
# Restrict scoring to the 20 standard amino acids
aa_string = "ACDEFGHIKLMNPQRSTVWY"
aa_list = list(aa_string)
aa_token_ids = [tokenizer.convert_tokens_to_ids(a) for a in aa_list]


@torch.no_grad()
def score_mutation(sequence, position_0idx, mut_aa):
    """Masked marginal log-likelihood ratio: log P(mut) - log P(wt)."""
    masked = (
        sequence[:position_0idx] + tokenizer.mask_token + sequence[position_0idx + 1 :]
    )  # Masked sequence at position position_0idx
    inputs = tokenizer(masked, return_tensors="pt").to(
        device
    )  # Convert masked sequence to integer IDs
    # And use PyTorch ("pt') and .to()

    logits = model(**inputs).logits  # Score of the model

    # Locate the mask token in the (tokenizer-specific) tokenized sequence
    mask_idx = (
        (inputs.input_ids == tokenizer.mask_token_id).nonzero()[0, 1].item()
    )  # Get position of the masked token

    # Restrict to 20 AAs and renormalize
    aa_logits = logits[
        0, mask_idx, aa_token_ids
    ]  # Get the score (logits) from the 20 amino acids (aa_token_ids)
    log_probs = torch.log_softmax(
        aa_logits, dim=-1
    )  # Apply log softmax to convert to log probabilities.

    wt_aa = sequence[position_0idx]  # Get the Wild type at position
    wt_log_p = log_probs[aa_list.index(wt_aa)].item()  # Get the log prob of the wild type
    mut_log_p = log_probs[aa_list.index(mut_aa)].item()  # Get the log prob of the mutant
    scored_mutation = (
        mut_log_p - wt_log_p
    )  # Scored mutation is the differences between mutant and wt, in log prob.
    return scored_mutation

### Score the key TEM-1 mutations

These six mutations span a range of known biological effects:

| Mutation | Known effect |
|---|---|
| **A42G** | Near-neutral |
| **R65L** | Mildly deleterious |
| **E104K** | Activity-enhancing, destabilizing |
| **W165R** | Strongly deleterious |
| **M182T** | Stabilizing, near-neutral activity — classic global suppressor |
| **G238S** | Activity-enhancing (extended-spectrum), but destabilizing |

In [ ]:
mutations_to_score = [
    ("H24S", 24, "H", "S"),
    ("A42G", 42, "A", "G"),
    ("R65L", 65, "R", "L"),
    ("E104K", 104, "E", "K"),
    ("W165R", 165, "W", "R"),
    ("M182T", 182, "M", "T"),
    ("G238S", 238, "G", "S"),
]

results = []
for name, ambler, wt, mut in mutations_to_score:
    # ambler_to_index handles the conversion from Ambler position to mature[idx]
    idx = ambler_to_index(ambler)
    assert TEM1_MATURE[idx] == wt, f"WT mismatch at {name}"
    score = score_mutation(TEM1_MATURE, idx, mut)
    results.append({"mutation": name, "esm2_score": score})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

**What to look for:**

- A42G is be near zero (mild change, surface-exposed)
- R65L is very destabilising.
- W165R should have the most negative score
- The activity-enhancing mutations (E104K, G238S) should be moderately negative — ESM-2 sees them as unusual because they're not common in evolution, even though they help under antibiotic selection.
- M182T is seen as strongly  positive — it is known to be a "fix-up" mutation that's structurally favorable.

This already shows the limitation of zero-shot scoring: ESM-2 captures *evolutionary plausibility*, which correlates with but isn't identical to *experimental fitness under specific conditions*.

## 5. Per-position constraint profile

For each position, mask it and compute the entropy of ESM-2's predicted distribution over the 20 amino acids:

$$H_i = -\sum_{a \in AA} P(a \mid \text{context}_i) \log P(a \mid \text{context}_i)$$

- **Low entropy** → ESM-2 strongly prefers one or two amino acids → constrained position
- **High entropy** → flat distribution → tolerant position

This profile should highlight the active site **without us telling the model anything about TEM-1's function**.

In [ ]:
@torch.no_grad()
def position_entropies(sequence):
    """Compute per-position entropy over the 20 amino acids. Returns shape (L,)."""
    entropies = np.zeros(len(sequence))
    for pos in range(len(sequence)):
        masked = sequence[:pos] + tokenizer.mask_token + sequence[pos + 1 :]
        inputs = tokenizer(masked, return_tensors="pt").to(device)
        logits = model(**inputs).logits

        mask_idx = (inputs.input_ids == tokenizer.mask_token_id).nonzero()[0, 1].item()
        aa_logits = logits[0, mask_idx, aa_token_ids]
        probs = torch.softmax(aa_logits, dim=-1).cpu().numpy()
        entropies[pos] = -np.sum(probs * np.log(probs + 1e-12))
    return entropies


print("Computing entropy at each position (few seconds on GPU/MPS, 1–2 min on CPU)…")
entropies = position_entropies(TEM1_MATURE)
print(f"Done. Range: {entropies.min():.2f} – {entropies.max():.2f} bits")

In [ ]:
ambler_positions = np.arange(26, 26 + len(TEM1_MATURE))

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(ambler_positions, entropies, color="steelblue", linewidth=1)
ax.fill_between(ambler_positions, entropies, alpha=0.25, color="steelblue")

for name, pos in ACTIVE_SITE.items():
    ax.axvline(pos, color="crimson", linestyle="--", alpha=0.7, linewidth=1)
    ax.text(
        pos,
        ax.get_ylim()[1] * 0.95,
        name,
        rotation=90,
        fontsize=9,
        color="crimson",
        ha="right",
        va="top",
    )

ax.set_xlabel("Position in TEM-1 (Ambler numbering)")
ax.set_ylabel("ESM-2 entropy (bits)")
ax.set_title("Per-position constraint — low values mark conserved residues")
ax.set_xlim(ambler_positions.min(), ambler_positions.max())
plt.tight_layout()
plt.show()

**Interpretation:**

- The dashed red lines mark known catalytic residues — most should sit in low-entropy regions
- Surface loops away from the active site appear as high-entropy peaks
- The catalytic residues S70, K73, S130, N132, and K234 fall in low-entropy regions. E166 in the omega loop is a notable exception — it's catalytically essential but sequence-flexible across β-lactamases, illustrating that ESM-2's entropy reflects evolutionary diversity, not mechanistic importance directly.
- ESM-2 was never told which residues are catalytic — this signal comes from evolutionary patterns it learned during pretraining

This is a great sanity check: the model has internalized the kind of functional constraint we'd otherwise extract from a multiple sequence alignment.

In [ ]:
@torch.no_grad()
def position_scan(sequence):
    """Compute per-position entropy over the 20 amino acids. Returns shape (L,)."""
    all_scores = []
    for pos in range(len(sequence)):
        masked = sequence[:pos] + tokenizer.mask_token + sequence[pos + 1 :]
        inputs = tokenizer(masked, return_tensors="pt").to(device)
        logits = model(**inputs).logits

        mask_idx = (inputs.input_ids == tokenizer.mask_token_id).nonzero()[0, 1].item()
        aa_logits = logits[0, mask_idx, aa_token_ids]
        all_log_p = torch.log_softmax(aa_logits, dim=-1).cpu().numpy()
        wt_aa = sequence[pos]
        wt_log_p = all_log_p[aa_list.index(wt_aa)]
        scored_mutation = all_log_p - wt_log_p
        all_scores.append(scored_mutation)
    return all_scores

In [ ]:
full_scan = position_scan(TEM1_MATURE)

In [ ]:
full_scan

In [ ]:
df_full_scan = pd.DataFrame(full_scan).T

In [ ]:
df_full_scan.index = aa_list
df_full_scan.columns = [str(x + 3 + SIGNAL_PEPTIDE_LENGTH) for x in range(len(TEM1_MATURE))]

In [ ]:
fig, ax = plt.subplots(figsize=(45, 4))
sns.heatmap(df_full_scan, cmap="Spectral_r", center=0, vmin=-10, vmax=10)
plt.title("ESM-2 zero-shot mutation preferences across TEM-1", y=1.15, fontsize=16)
ax.tick_params(axis="y", labelrotation=0)
ax.set_xlabel(
    "Position in TEM-1 (Ambler numbering) - black dots mark wild-type - active site positions highlighted"
)
ax.set_ylabel("Mutated to")
ax.tick_params(axis="y", labelrotation=0)
plt.tight_layout()
# Mark the wild-type residue at each position with a small dot
for pos in range(len(TEM1_MATURE)):
    wt_aa = TEM1_MATURE[pos]
    wt_row = aa_list.index(wt_aa)
    ax.scatter(pos + 0.5, wt_row + 0.5, marker=".", color="black", s=8)

# Mark catalytic residues and binding sites
for name, pos in ACTIVE_SITE.items():
    if str(pos) in df_full_scan.columns:
        x = df_full_scan.columns.index(str(pos)) - 2 + 0.5
        ax.axvline(x, color="black", linestyle="--", linewidth=0.8, alpha=0.6)
        ax.text(x, -0.5, name, rotation=90, fontsize=8, ha="center", va="bottom")

fig.savefig("heatmap.png", dpi=150, bbox_inches="tight")

Figure: Per-position ESM-2 mutation preferences across TEM-1. Each cell shows log P(mutant) − log P(wild-type) at that position, with red indicating substitutions ESM-2 prefers over wild-type and blue indicating disfavored substitutions. Black dots mark the wild-type residue at each position (where the score is zero by definition). The catalytic residues (S70, K73, S130, N132, E166, K234) appear as deep-blue vertical stripes — every alternative substitution is disfavored, indicating these positions are essentially irreplaceable from an evolutionary perspective. The substrate-binding triad K234-S235-G236 shows the same pattern. ESM-2 was never told these residues are catalytically important; this signal comes purely from evolutionary patterns the model learned during pretraining.

## 6. Compare to experimental fitness (Firnberg et al. 2014)

We'll download the deep mutational scanning dataset from ProteinGym and check how well our zero-shot ESM-2 scores correlate with experimental fitness.

In [ ]:
# File mirrored in this repo from ProteinGym v1.3
# Original source: https://github.com/OATML-Markslab/ProteinGym
dms = pd.read_csv("data/BLAT_ECOLX_Firnberg_2014.csv")
print(f"Loaded {len(dms)} mutations")

In [ ]:
# Parse the mutant string (e.g., "M182T") into wt, position, mut
def parse_mutant(s):
    m = re.match(r"([A-Z])(\d+)([A-Z])", s)
    if not m:
        return None
    wt, pos, mut = m.group(1), int(m.group(2)), m.group(3)
    return wt, pos, mut

ProteinGym CSVs typically have columns `mutant` (e.g., `M180T`), `mutated_sequence`, and `DMS_score`. We need to extract the position and convert it to Ambler position

In [ ]:
dms[["wt", "pos", "mut"]] = dms["mutant"].apply(parse_mutant).tolist()
dms["ambler_pos"] = dms["pos"] + 2
dms["ambler_mutant"] = dms["wt"] + dms["ambler_pos"].astype(str) + dms["mut"]

We will score each single mutation with ESM-2 and compute the Spearman correlation with the experimental score.

To keep this notebook fast, we score a random subsample of 300 mutations. Increase the sample size if you have time.

In [ ]:
# Sample for speed
sample = dms.sample(n=min(300, len(dms)), random_state=42).reset_index(drop=True)
esm_scores = []
exp_scores = []

for _, row in sample.iterrows():
    idx = ambler_to_index(row["ambler_pos"])
    if idx < 0 or idx >= len(TEM1_MATURE) or TEM1_MATURE[idx] != row["wt"]:
        continue
    s = score_mutation(TEM1_MATURE, idx, row["mut"])
    esm_scores.append(s)
    exp_scores.append(row["DMS_score"])

print(f"Scored {len(esm_scores)} mutations")

In [ ]:
rho, pval = spearmanr(esm_scores, exp_scores)
print(f"Spearman ρ between ESM-2 and experimental fitness: {rho:.3f}  (p = {pval:.2e})")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(esm_scores, exp_scores, alpha=0.5, s=20)
ax.set_xlabel("ESM-2 zero-shot score (log-likelihood ratio)")
ax.set_ylabel("Experimental fitness (Firnberg 2014)")
ax.set_title(f"Zero-shot ESM-2 vs experiment — Spearman ρ = {rho:.3f}")
ax.axhline(0, color="grey", linewidth=0.5)
ax.axvline(0, color="grey", linewidth=0.5)
plt.tight_layout()
plt.show()

Spearman ρ around 0.5 is typical for ESM-2 35M on TEM-1 (the larger 650M model often reaches 0.55+). That's a real signal — but many mutations fall far from the diagonal, and the scores aren't calibrated to fitness units.

## 7. Takeaways

**What worked:**
- ESM-2 distinguishes deleterious from tolerated mutations from sequence alone
- The entropy profile recovers known catalytic residues without supervision
- Useful for **ranking** mutations when you have no labeled data

**What didn't:**
- Scores aren't calibrated to fitness or stability units
- Some functionally important mutations look "unusual" to the model (low evolutionary frequency ≠ low fitness in your assay)
- Performance plateaus around Spearman ρ ≈ 0.5-0.55 for TEM-1

**When to use zero-shot scoring:**
- You have no labeled data
- You want to quickly screen many proteins
- You only need rankings, not absolute predictions
- You're studying a protein where DMS data isn't available

**On the model size:**
- With the 35M model, you may see some biologically obvious destabilizing mutations score positive — the smaller model relies more heavily on local sequence context. Switch to esm2_t33_650M_UR50D for sharper predictions if you have a GPU


When you *do* have labeled data, fine-tuning can push performance considerably higher — that's what we'll do in [Part 2](./02_fine_tuning.ipynb).